## Without web search

In [1]:
import os
from dotenv import load_dotenv

# Force reload the .env file, overwriting the old cached variables
load_dotenv(override=True)
print("Tracing Enabled:", os.environ.get("LANGSMITH_TRACING"))
print("Project Name:", os.environ.get("LANGSMITH_PROJECT"))
# Make sure your API key is loaded (don't print the whole thing)
api_key = os.environ.get("LANGSMITH_API_KEY")
print("API Key Loaded:", bool(api_key))

Tracing Enabled: true
Project Name: lca-lc-foundation
API Key Loaded: True


In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.5,
)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
)

/Users/vikas4mac/Documents/workspace/agentic_ai/langchain_qs/lca-lc-foundations/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

My training data has a **knowledge cutoff of July 2024**. This means:

- I was trained on a vast dataset of text and code **up to and including July 2024**.
- I do **not** have access to real-time information, live internet searches, or events that occurred after July 2024.
- For topics, events, statistics, or developments **after July 2024**, my knowledge may be incomplete, outdated, or absent.

### Important Notes:
- **Not "real-time"**: I cannot browse the web or check current news, stock prices, weather, etc.
- **Cutoff ≠ Exact Date**: The cutoff means my training data *stops being updated* around that time. I may not have comprehensive coverage of *every* event right up to July 31, 2024, but my knowledge generally reflects the state of the world up to that period.
- **Reason for Cutoff**: This is a standard practice for large language models to ensure training stability, data quality, and to avoid incorporating potentially unverified or rapidly changing information too close to de

## Add web search tool

In [6]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current Chief Minister of Delhi, India?")

{'query': 'Who is the current Chief Minister of Delhi, India?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Rekha_Gupta',
   'title': 'Rekha Gupta - Wikipedia',
   'content': 'Rekha Gupta (born 19 July 1974) is an Indian politician who is serving as the current Chief Minister of Delhi from February 2025. A member of Bharatiya',
   'score': 0.93425745,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Chief_Minister_of_Delhi',
   'title': 'Chief Minister of Delhi - Wikipedia',
   'content': 'Rekha Gupta is the incumbent chief minister since February 2025. Chief Minister of National Capital Territory of Delhi. Emblem of the NCT of Delhi. Flag of',
   'score': 0.9237645,
   'raw_content': None},
  {'url': 'https://www.britannica.com/topic/List-of-current-Indian-chief-ministers',
   'title': 'List of current Indian chief ministers - Britannica',
   'content': 'Vishnu Deo Sai, Chhattisgarh, BJP, December 1

In [7]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current Chief Minister of Delhi, India?")

response = agent.invoke(
    {"messages": [question]}
)

In [8]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current Chief Minister of Delhi, India?', additional_kwargs={}, response_metadata={}, id='79ab6061-f0d5-4329-abd1-dca026dfbce7'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking: "Who is the current Chief Minister of Delhi, India?" I need to provide the current Chief Minister of Delhi. As of my knowledge cutoff in 2024, the Chief Minister of Delhi is Arvind Kejriwal. However, I should verify if there have been any recent changes. I\'ll use the web search tool to get the most up-to-date information.\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking: "Who is the current Chief Minister of Delhi, India?" I need to provide the current Chief Minister of Delhi. As of my knowledge cutoff in 2024, the Chief Minister of Delhi is Arvind Kejriwal. However, I should verify if there have been any recent changes. I\'ll use the web search tool to get the most up-to-date 

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r